In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "healthcare_analytics"
SILVER = "silver"
GOLD = "gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD}")


def silver_table(name):
    return spark.table(f"{CATALOG}.{SILVER}.{name}")


def save_gold(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{GOLD}.{table_name}")
    )
    print(f"Created {CATALOG}.{GOLD}.{table_name}")


def surrogate_key(column_name):
    return F.when(
        F.col(column_name).isNull(),
        F.lit(None)
    ).otherwise(
        F.sha2(F.col(column_name).cast("string"), 256)
    )


def existing_columns(df, columns):
    return [c for c in columns if c in df.columns]

In [0]:
# ============================================================
# DIM PATIENT
# ============================================================

patients = silver_table("patients")

# Correct age logic for deceased patients:
# age should stop increasing at death date.
if "birth_date" in patients.columns:
    patients = patients.withColumn(
        "age_at_reference",
        F.floor(
            F.months_between(
                F.coalesce(
                    F.col("death_date"),
                    F.current_date()
                ),
                F.col("birth_date")
            ) / 12
        )
    )

patient_columns = existing_columns(
    patients,
    [
        "patient_id",
        "patient_name",
        "birth_date",
        "death_date",
        "age_at_reference",
        "age_band",
        "gender",
        "race",
        "ethnicity",
        "marital",
        "city",
        "state",
        "county",
        "zip",
        "lat",
        "lon",
        "healthcare_expenses",
        "healthcare_coverage",
        "income",
        "is_deceased"
    ]
)

dim_patient = (
    patients
    .select(*patient_columns)
    .withColumn(
        "patient_key",
        surrogate_key("patient_id")
    )
)

save_gold(dim_patient, "dim_patient")


# ============================================================
# DIM PROVIDER
# ============================================================

providers = silver_table("providers")

provider_columns = existing_columns(
    providers,
    [
        "provider_id",
        "provider_name",
        "organization_id",
        "gender",
        "speciality",
        "specialty",
        "address",
        "city",
        "state",
        "zip",
        "lat",
        "lon",
        "utilization"
    ]
)

dim_provider = (
    providers
    .select(*provider_columns)
    .withColumn(
        "provider_key",
        surrogate_key("provider_id")
    )
)

save_gold(dim_provider, "dim_provider")


# ============================================================
# DIM FACILITY
# ============================================================

organizations = silver_table("organizations")

facility_columns = existing_columns(
    organizations,
    [
        "organization_id",
        "organization_name",
        "address",
        "city",
        "state",
        "zip",
        "lat",
        "lon",
        "phone",
        "revenue",
        "utilization"
    ]
)

dim_facility = (
    organizations
    .select(*facility_columns)
    .withColumn(
        "facility_key",
        surrogate_key("organization_id")
    )
)

save_gold(dim_facility, "dim_facility")


# ============================================================
# DIM PAYER
# ============================================================

payers = silver_table("payers")

payer_columns = existing_columns(
    payers,
    [
        "payer_id",
        "payer_name",
        "ownership",
        "address",
        "city",
        "state_headquartered",
        "zip",
        "phone",
        "amount_covered",
        "amount_uncovered",
        "revenue",
        "covered_encounters",
        "uncovered_encounters",
        "unique_customers",
        "member_months"
    ]
)

dim_payer = (
    payers
    .select(*payer_columns)
    .withColumn(
        "payer_key",
        surrogate_key("payer_id")
    )
)

save_gold(dim_payer, "dim_payer")


# ============================================================
# DIM DIAGNOSIS
# ============================================================

conditions = silver_table("conditions")

diagnosis_columns = existing_columns(
    conditions,
    ["system", "code", "description"]
)

dim_diagnosis = (
    conditions
    .select(*diagnosis_columns)
    .dropDuplicates()
    .withColumn(
        "diagnosis_key",
        F.sha2(
            F.concat_ws(
                "|",
                F.coalesce(F.col("system"), F.lit("")),
                F.coalesce(F.col("code"), F.lit("")),
                F.coalesce(F.col("description"), F.lit(""))
            ),
            256
        )
    )
)

save_gold(dim_diagnosis, "dim_diagnosis")


# ============================================================
# DIM DATE
# ============================================================

encounters = silver_table("encounters")

bounds = (
    encounters
    .select(
        F.min(F.to_date("encounter_start")).alias("min_date"),
        F.max(F.to_date("encounter_start")).alias("max_date")
    )
    .first()
)

date_range = spark.createDataFrame(
    [(bounds["min_date"], bounds["max_date"])],
    ["min_date", "max_date"]
)

dim_date = (
    date_range
    .select(
        F.explode(
            F.sequence(
                F.col("min_date"),
                F.col("max_date")
            )
        ).alias("date")
    )
    .withColumn(
        "date_key",
        F.date_format("date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month_number", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("day_of_month", F.dayofmonth("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn(
        "is_weekend",
        F.dayofweek("date").isin([1, 7])
    )
)

save_gold(dim_date, "dim_date")

Created healthcare_analytics.gold.dim_patient
Created healthcare_analytics.gold.dim_provider
Created healthcare_analytics.gold.dim_facility
Created healthcare_analytics.gold.dim_payer
Created healthcare_analytics.gold.dim_diagnosis
Created healthcare_analytics.gold.dim_date


In [0]:
# ============================================================
# FACT ENCOUNTER
# ============================================================

encounters = silver_table("encounters")

fact_encounter = (
    encounters
    .withColumn(
        "encounter_key",
        surrogate_key("encounter_id")
    )
    .withColumn(
        "patient_key",
        surrogate_key("patient_id")
    )
    .withColumn(
        "provider_key",
        surrogate_key("provider_id")
    )
    .withColumn(
        "facility_key",
        surrogate_key("organization_id")
    )
    .withColumn(
        "payer_key",
        surrogate_key("payer_id")
    )
    .withColumn(
        "encounter_date_key",
        F.date_format(
            F.to_date("encounter_start"),
            "yyyyMMdd"
        ).cast("int")
    )
)

save_gold(fact_encounter, "fact_encounter")


# ============================================================
# FACT PROCEDURE
# ============================================================

procedures = silver_table("procedures")

fact_procedure = (
    procedures
    .withColumn(
        "procedure_key",
        F.sha2(
            F.concat_ws(
                "|",
                F.coalesce(F.col("patient_id"), F.lit("")),
                F.coalesce(F.col("encounter_id"), F.lit("")),
                F.coalesce(F.col("code"), F.lit("")),
                F.coalesce(
                    F.col("procedure_start").cast("string"),
                    F.lit("")
                ),
                F.coalesce(F.col("description"), F.lit(""))
            ),
            256
        )
    )
    .withColumn(
        "patient_key",
        surrogate_key("patient_id")
    )
    .withColumn(
        "encounter_key",
        surrogate_key("encounter_id")
    )
)

save_gold(fact_procedure, "fact_procedure")


# ============================================================
# FACT CLAIM
# ============================================================

claims = silver_table("claims")

fact_claim = (
    claims
    .withColumn(
        "claim_key",
        surrogate_key("claim_id")
    )
    .withColumn(
        "patient_key",
        surrogate_key("patient_id")
    )
    .withColumn(
        "provider_key",
        surrogate_key("provider_id")
    )
)

if "primary_payer_id" in fact_claim.columns:
    fact_claim = fact_claim.withColumn(
        "primary_payer_key",
        surrogate_key("primary_payer_id")
    )

if "secondary_payer_id" in fact_claim.columns:
    fact_claim = fact_claim.withColumn(
        "secondary_payer_key",
        surrogate_key("secondary_payer_id")
    )

save_gold(fact_claim, "fact_claim")

Created healthcare_analytics.gold.fact_encounter
Created healthcare_analytics.gold.fact_procedure
Created healthcare_analytics.gold.fact_claim


In [0]:
encounters = silver_table("encounters")

inpatient = (
    encounters
    .filter(
        F.lower(F.col("encounter_class")) == "inpatient"
    )
    .select(
        "encounter_id",
        "patient_id",
        "organization_id",
        "provider_id",
        "payer_id",
        "encounter_start",
        "encounter_end",
        "total_claim_cost",
        "payer_coverage",
        "patient_responsibility"
    )
)

patient_window = (
    Window
    .partitionBy("patient_id")
    .orderBy("encounter_start")
)

readmissions = (
    inpatient
    .withColumn(
        "next_inpatient_encounter_id",
        F.lead("encounter_id").over(patient_window)
    )
    .withColumn(
        "next_inpatient_start",
        F.lead("encounter_start").over(patient_window)
    )
    .withColumn(
        "days_to_readmission",
        F.datediff(
            F.to_date("next_inpatient_start"),
            F.to_date("encounter_end")
        )
    )
    .withColumn(
        "readmission_30d_flag",
        F.when(
            (F.col("days_to_readmission") >= 0)
            & (F.col("days_to_readmission") <= 30),
            1
        ).otherwise(0)
    )
    .withColumn(
        "index_encounter_key",
        surrogate_key("encounter_id")
    )
    .withColumn(
        "patient_key",
        surrogate_key("patient_id")
    )
    .withColumn(
        "facility_key",
        surrogate_key("organization_id")
    )
    .withColumn(
        "provider_key",
        surrogate_key("provider_id")
    )
    .withColumn(
        "payer_key",
        surrogate_key("payer_id")
    )
)

save_gold(
    readmissions,
    "fact_readmission"
)

display(
    readmissions
    .groupBy("readmission_30d_flag")
    .count()
)

Created healthcare_analytics.gold.fact_readmission


readmission_30d_flag,count
0,805
1,188


In [0]:
gold_tables = spark.sql("""
SHOW TABLES IN healthcare_analytics.gold
""")

display(gold_tables)

print(
    "Gold core table count:",
    gold_tables.count()
)

database,tableName,isTemporary
gold,dim_date,false
gold,dim_diagnosis,false
gold,dim_facility,false
gold,dim_patient,false
gold,dim_payer,false
gold,dim_provider,false
gold,fact_claim,false
gold,fact_encounter,false
gold,fact_procedure,false
gold,fact_readmission,false


Gold core table count: 10


In [0]:
qa_results = []

for row in gold_tables.collect():
    table_name = row["tableName"]

    df = spark.table(
        f"healthcare_analytics.gold.{table_name}"
    )

    qa_results.append(
        (
            table_name,
            df.count(),
            len(df.columns)
        )
    )

qa_df = spark.createDataFrame(
    qa_results,
    [
        "table_name",
        "row_count",
        "column_count"
    ]
)

display(
    qa_df.orderBy(F.desc("row_count"))
)

table_name,row_count,column_count
fact_procedure,172791,17
fact_claim,114186,39
fact_encounter,63795,29
dim_date,38271,10
dim_patient,1138,21
fact_readmission,993,19
dim_facility,827,12
dim_provider,827,12
dim_diagnosis,264,4
dim_payer,10,16
